# 01 · Tái hiện kết quả cú sốc giá của ICTA2026

Notebook này chạy lại thí nghiệm **cú sốc giá ×2 tại tập 300** bằng mã gốc của bài ICTA2026
(`icta2026/sr_plr_marl_v8_mc_mr.py`, được gọi qua `supplementary/ext.py`) trên mạng đơn hàng hóa $2\times2$ và $3\times3$.

* Tiêu chí phục hồi: **$G_{50}$** (không phụ thuộc $D_{\max}$ riêng của bộ điều phối): phục hồi ở tập đầu tiên sau đỉnh mà $G_\lambda\le 0{,}5\,G_{\rm peak}$.
* `SEEDS=30` tái hiện đúng bảng trong báo cáo (mất vài phút); đặt `SEEDS=5` để chạy nhanh.
* Lưu ý trung thực: theo tiêu chí này **Fixed-LR phục hồi nhanh hơn SR-PLR-MARL**; ưu thế của SR-PLR-MARL là so với PLR gốc và Poly-decay (xem Mục 4.7 của báo cáo).

In [1]:
# --- setup: chạy được cả trên Colab lẫn máy cục bộ ---
import os, sys, subprocess
REPO_URL = "https://github.com/hieuthanhpham12/marl-vi-trade-crisis.git"
if not os.path.isdir("icta2026"):
    if os.path.isdir("../icta2026"):
        os.chdir("..")                                   # đang chạy trong notebooks/
    else:
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir("marl-vi-trade-crisis")
for sub in ("icta2026", "supplementary", "marl_baselines"):
    sys.path.insert(0, os.path.abspath(sub))
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
print("thư mục làm việc:", os.getcwd())

thư mục làm việc: /content/marl-vi-trade-crisis


In [2]:
import ext
SEEDS = 5          # đặt 30 để tái hiện bảng của báo cáo
METHODS = ["PLR-original", "SR-PLR-MARL", "Fixed-LR", "Poly-decay", "PG-fixed", "EG"]
M = ext.schedulers()
rows, traj = [], {}
for (m, n) in [(2, 2), (3, 3)]:
    for name in METHODS:
        ok, rts, Gf = [], [], []
        for s in range(SEEDS):
            net = ext.make_single(m, n)
            r = ext.run_ext(M[name](), net, s, N=800, shocks=((300, "price", 2.0),), sigma=0.01)
            a, b = ext.g50(r["G"], 300, 800); ok.append(a); rts.append(b); Gf.append(r["G"][-1])
            if s == 0: traj[(m, n, name)] = r["G"]
        rows.append(dict(mạng=f"{m}x{n}", phương_pháp=name, Succ=np.mean(ok), RT_G50=np.mean(rts), RT_sd=np.std(rts, ddof=1) if SEEDS > 1 else 0.0, G_cuối=np.mean(Gf)))
pd.DataFrame(rows).round(3)

,mạng,phương_pháp,Succ,RT_G50,RT_sd,G_cuối
0,2x2,PLR-original,0.0,400.0,0.000,6.819
1,2x2,SR-PLR-MARL,1.0,240.0,0.000,4.489
2,2x2,Fixed-LR,1.0,159.0,0.000,1.197
3,2x2,Poly-decay,0.0,400.0,0.000,14.670
4,2x2,PG-fixed,1.0,17.0,0.000,0.003
5,2x2,EG,1.0,18.0,0.000,0.003
6,3x3,PLR-original,1.0,361.0,0.000,6.585
7,3x3,SR-PLR-MARL,1.0,144.8,0.447,4.176
8,3x3,Fixed-LR,1.0,110.0,0.000,0.561
9,3x3,Poly-decay,0.0,400.0,0.000,19.059


In [3]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (m, n) in zip(axs, [(2, 2), (3, 3)]):
    for name in METHODS:
        ax.semilogy(np.maximum(traj[(m, n, name)], 1e-4), label=name, lw=1.2)
    ax.axvline(300, color="k", ls=":", lw=0.8); ax.set_title(f"Mạng {m}×{n}, hạt giống 0"); ax.set_xlabel("tập"); ax.grid(alpha=0.3)
axs[0].set_ylabel(r"phần dư $G_\lambda(x_t)$"); axs[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Cách đọc.** PLR gốc và Poly-decay giữ bước nhỏ sau cú sốc nên $G_\lambda$ giảm rất chậm; SR-PLR-MARL phát hiện cú sốc, đặt lại đồng hồ cục bộ và phục hồi nhanh hơn rõ rệt;
PG/EG (bước $\alpha=0{,}1$ đã chỉnh) và Fixed-LR phục hồi nhanh hơn SR-PLR-MARL.